# 03 EOT20 tidal uncertainty

**Manuscript outputs:** Table A1 (EOT20 validation); annual U_eta used for Table A2

**Inputs:** EOT20 constituents, Mombasa hourly gauge, scene acquisition times

Inputs are read from `Master_raw/`; everything produced is written to `Outputs/` (manuscript figures in `Outputs/Figures`, tables in `Outputs/Tables`). The first code cell lists every path. Run the cells in order.

# EOT20 Validation and Tidal Input Preparation

**Author:** James Mbugua  
**Registration:** ENC324-1769/2022

## Purpose

This notebook does three things only:

1. quality-controls the Mombasa PSMSL high-frequency tide-gauge record;
2. validates EOT20 against the gauge record; and
3. predicts EOT20 at Landsat acquisition times and prepares the annual vertical tidal input used later in the horizontal tidal uncertainty calculation.

The annual output is clearly marked as the **Mombasa tidal input**. It is not yet a coast-wide \(U_{tide}\) product because the current EOT20 predictions are made only at the Mombasa validation location.

## 1. Imports and project paths

In [1]:
from pathlib import Path

# ------------------------------------------------------------
# REPOSITORY ROOT: first folder above this notebook that contains Master_raw
# (e.g. D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024)
# ------------------------------------------------------------
_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents) if (p / "Master_raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No Master_raw folder found above {_here}. Open this notebook inside the repository.")
RAW = ROOT / "Master_raw"          # inputs
OUTPUTS = ROOT / "Outputs"         # everything this repository produces
FIGURES = OUTPUTS / "Figures"      # manuscript figures
TABLES = OUTPUTS / "Tables"        # manuscript tables
TEXT_STATS = TABLES / "Text_statistics"
for _d in (FIGURES, TABLES, TEXT_STATS):
    _d.mkdir(parents=True, exist_ok=True)
OVERWRITE = True                   # re-running replaces this notebook's earlier outputs


def first_existing(*candidates):
    """Return the first path that exists (otherwise the first one)."""
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return Path(candidates[0])


def save_table(df, name, index=False, folder=None):
    path = (folder or TABLES) / f"{name}.csv"
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print("Saved:", path.relative_to(ROOT))


def save_figure(fig, name, dpi=600):
    for ext in ("jpg",):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=dpi, bbox_inches="tight", pil_kwargs=({"quality": 95} if ext == "jpg" else None))
    print("Saved:", (FIGURES / f"{name}.jpg").relative_to(ROOT))


def remove_existing(*files):
    for f in map(Path, files):
        for t in (list(f.parent.glob(f.stem + ".*")) if f.suffix == ".shp" else [f]):
            if t.exists():
                if not OVERWRITE:
                    raise FileExistsError(t)
                t.unlink()

# ------------------------------------------------------------
# INPUTS AND OUTPUT FOLDERS FOR THIS NOTEBOOK
# ------------------------------------------------------------
EOT20_DIR = RAW / "EOT20_tides"
TIDE_GAUGE_DIR = RAW / "Tide_Gauge_PSMSL"
TIDE_GAUGE_CSV = TIDE_GAUGE_DIR / "PMSL_Hourly_sealevel_data_mombasa.csv"
SCENE_LOG_DIR = RAW / "Satellite_Coastlines_GEE" / "Sensor log"
SCENE_METADATA_CSV = SCENE_LOG_DIR / "scene_metadata_full_2000_2024.csv"
SCENE_TIMES_CSV = SCENE_LOG_DIR / "compiled_scene_metadata_timestamp_2000_2024.csv"
EOT20_OUT_DIR = OUTPUTS / "EOT20_PMSL_tide"
EOT20_VALIDATION_CSV = EOT20_OUT_DIR / "Mombasa_EOT20_validation_summary.csv"
U_ETA_CSV = EOT20_OUT_DIR / "Utide_Vertical_Tide_Input_Mombasa.csv"

# Scene acquisition times. If the compiled timestamp file is not in the repository,
# build the same columns from scene_metadata_full_2000_2024.csv (one row per scene).
if not SCENE_TIMES_CSV.exists() and SCENE_METADATA_CSV.exists():
    import pandas as _pd
    _m = _pd.read_csv(SCENE_METADATA_CSV)
    _t = _pd.DataFrame({"year": _m["year"], "sensor": _m["sensor"], "scene_id": _m["landsat_product_id"],
                        "acquisition_datetime": _m["time_start_iso"], "cloud_cover": _m["cloud_cover"]})
    EOT20_OUT_DIR.mkdir(parents=True, exist_ok=True)
    SCENE_TIMES_CSV = EOT20_OUT_DIR / "scene_times_from_scene_metadata_full.csv"
    _t.to_csv(SCENE_TIMES_CSV, index=False)
    print(f"Scene times built from {SCENE_METADATA_CSV.name}: {len(_t):,} scenes -> {SCENE_TIMES_CSV.relative_to(ROOT)}")

print("Repository root:", ROOT)
for _label, _p in [("EOT20_DIR", EOT20_DIR), ("TIDE_GAUGE_CSV", TIDE_GAUGE_CSV), ("SCENE_TIMES_CSV", SCENE_TIMES_CSV)]:
    print(f"  {'ok     ' if _p.exists() else 'MISSING'} {_label}: {_p.relative_to(ROOT)}")

from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pyTMD

warnings.filterwarnings("ignore", category=FutureWarning)
GAUGE_DIR = TIDE_GAUGE_DIR
SCENE_METADATA_FILE = SCENE_TIMES_CSV

OUTPUT_DIR = EOT20_OUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MOMBASA_LON = 39.6567
MOMBASA_LAT = -4.07

# Final tables retained from this notebook
VALIDATION_SUMMARY_OUT = EOT20_VALIDATION_CSV
UTIDE_VERTICAL_INPUT_OUT = U_ETA_CSV

print("Output directory:", OUTPUT_DIR)

Scene times built from scene_metadata_full_2000_2024.csv: 5,332 scenes -> Outputs\EOT20_PMSL_tide\scene_times_from_scene_metadata_full.csv
Repository root: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024
  ok      EOT20_DIR: Master_raw\EOT20_tides
  ok      TIDE_GAUGE_CSV: Master_raw\Tide_Gauge_PSMSL\PMSL_Hourly_sealevel_data_mombasa.csv
  ok      SCENE_TIMES_CSV: Outputs\EOT20_PMSL_tide\scene_times_from_scene_metadata_full.csv
Output directory: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\EOT20_PMSL_tide


## 2. Load Landsat scene metadata

The metadata must contain one row per Landsat scene contributing to the annual shoreline composites. The acquisition timestamp is required because a median spectral composite does not have one unique acquisition time.

In [2]:
REQUIRED_SCENE_COLUMNS = [
    "year",
    "sensor",
    "scene_id",
    "acquisition_datetime",
    "cloud_cover",
]

scenes = pd.read_csv(SCENE_METADATA_FILE)

missing = [
    c for c in REQUIRED_SCENE_COLUMNS
    if c not in scenes.columns
]
if missing:
    raise KeyError(
        "Missing scene metadata columns: "
        + ", ".join(missing)
    )

scenes = scenes[REQUIRED_SCENE_COLUMNS].copy()
scenes["year"] = pd.to_numeric(
    scenes["year"], errors="coerce"
)
scenes["cloud_cover"] = pd.to_numeric(
    scenes["cloud_cover"], errors="coerce"
)
scenes["acquisition_datetime"] = pd.to_datetime(
    scenes["acquisition_datetime"],
    errors="coerce",
    utc=True
)

scenes = (
    scenes
    .dropna(subset=["year", "scene_id", "acquisition_datetime"])
    .drop_duplicates(subset="scene_id")
    .sort_values("acquisition_datetime")
    .reset_index(drop=True)
)

scenes["year"] = scenes["year"].astype(int)

print("Valid scene records:", f"{len(scenes):,}")
print("Years:", scenes["year"].min(), "to", scenes["year"].max())
display(
    scenes.groupby("year")
    .size()
    .rename("scene_count")
    .to_frame()
)

Valid scene records: 5,332
Years: 2000 to 2024


,scene_count
year,
2000,81
2001,88
2002,79
2003,77
2004,84
2005,91
2006,69
2007,98
2008,95


## 3. Load and quality-control the Mombasa PSMSL record

The source CSV is preserved unchanged. Primary validation uses only observations with QC flag 1. Missing-value codes and invalid timestamps are removed.

The Mombasa gauge datum is documented by PSMSL. However, the PSMSL RLR and EOT20 model reference surface are not assumed to be directly equivalent, so the observed and modelled series are centred independently for validation.

In [3]:
EXPECTED_GAUGE_COLUMNS = [
    "Date",
    "Time",
    "Sea_Level_m",
    "QC_Flag",
]

gauge_files = [TIDE_GAUGE_CSV]

if len(gauge_files) != 1:
    raise RuntimeError(
        "Expected exactly one tide-gauge CSV in "
        f"{GAUGE_DIR}, found {len(gauge_files)}."
    )

GAUGE_FILE = gauge_files[0]
raw_gauge = pd.read_csv(GAUGE_FILE)

missing = [
    c for c in EXPECTED_GAUGE_COLUMNS
    if c not in raw_gauge.columns
]
if missing:
    raise KeyError(
        "Missing gauge columns: "
        + ", ".join(missing)
    )

gauge = raw_gauge.copy()

gauge["datetime"] = pd.to_datetime(
    gauge["Date"].astype(str).str.strip()
    + " "
    + gauge["Time"].astype(str).str.strip(),
    errors="coerce",
    utc=True
)

gauge["observed_level"] = pd.to_numeric(
    gauge["Sea_Level_m"],
    errors="coerce"
)

gauge["qc_flag"] = pd.to_numeric(
    gauge["QC_Flag"],
    errors="coerce"
)

gauge = gauge.loc[
    (gauge["qc_flag"] == 1)
    & gauge["datetime"].notna()
    & gauge["observed_level"].notna()
    & (gauge["observed_level"] > -90),
    ["datetime", "observed_level", "qc_flag"]
].copy()

gauge = (
    gauge
    .sort_values("datetime")
    .drop_duplicates(subset="datetime")
    .reset_index(drop=True)
)

gaps_h = (
    gauge["datetime"]
    .diff()
    .dt.total_seconds()
    / 3600
)

print("Gauge file:", GAUGE_FILE.name)
print("Valid QC=1 observations:", f"{len(gauge):,}")
print("Start:", gauge["datetime"].min())
print("End:", gauge["datetime"].max())
print("Median interval (h):", gaps_h.median())
print("Gaps >1.5 h:", int((gaps_h > 1.5).sum()))

Gauge file: PMSL_Hourly_sealevel_data_mombasa.csv
Valid QC=1 observations: 84,034
Start: 1986-06-17 10:00:00+00:00
End: 1998-12-31 23:00:00+00:00
Median interval (h): 1.0
Gaps >1.5 h: 39


## 4. Configure EOT20 and prediction function

The Mombasa gauge coordinate falls outside the valid EOT20 ocean mask. The nearest valid EOT20 cell previously identified is approximately 12.02 km away, so a 20 km coastal extrapolation cutoff is retained for the Mombasa validation.

In [4]:
EXPECTED_EOT20_FILES = [
    "2N2_ocean_eot20.nc", "J1_ocean_eot20.nc",
    "K1_ocean_eot20.nc", "K2_ocean_eot20.nc",
    "M2_ocean_eot20.nc", "M4_ocean_eot20.nc",
    "MF_ocean_eot20.nc", "MM_ocean_eot20.nc",
    "N2_ocean_eot20.nc", "O1_ocean_eot20.nc",
    "P1_ocean_eot20.nc", "Q1_ocean_eot20.nc",
    "S1_ocean_eot20.nc", "S2_ocean_eot20.nc",
    "SA_ocean_eot20.nc", "SSA_ocean_eot20.nc",
    "T2_ocean_eot20.nc",
]

missing = [
    f for f in EXPECTED_EOT20_FILES
    if not (EOT20_DIR / f).exists()
]
if missing:
    raise FileNotFoundError(
        "Missing EOT20 files:\n" + "\n".join(missing)
    )

MODEL_ROOT = EOT20_DIR / "_pytmd_model"
MODEL_DIR = MODEL_ROOT / "EOT20" / "ocean_tides"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for name in EXPECTED_EOT20_FILES:
    src = EOT20_DIR / name
    dst = MODEL_DIR / name

    if dst.exists():
        continue

    try:
        os.link(src, dst)
    except OSError:
        try:
            os.symlink(src, dst)
        except OSError as exc:
            raise OSError(
                f"Could not link {name}: {exc}"
            )

def predict_eot20(lon, lat, datetimes):
    dt = pd.DatetimeIndex(
        pd.to_datetime(datetimes, utc=True)
    )

    tide = pyTMD.compute.tide_elevations(
        x=np.full(len(dt), float(lon)),
        y=np.full(len(dt), float(lat)),
        delta_time=dt.tz_convert(None).to_numpy(
            dtype="datetime64[ns]"
        ),
        directory=MODEL_ROOT,
        model="EOT20",
        crs=4326,
        type="trajectory",
        standard="datetime",
        method="linear",
        extrapolate=True,
        cutoff=20.0,
    )

    return np.ma.filled(
        np.ma.asarray(tide),
        np.nan
    ).astype(float)

print("pyTMD version:", pyTMD.__version__)
print("EOT20 model directory:", MODEL_DIR)

pyTMD version: 3.0.9
EOT20 model directory: D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Master_raw\EOT20_tides\_pytmd_model\EOT20\ocean_tides


## 5. Validate EOT20 against the Mombasa gauge

Observed and predicted series are centred independently before comparison. This preserves tidal variability while removing the unmatched absolute vertical offsets between the local gauge datum and the EOT20 reference surface.

In [5]:
validation = gauge.copy()

validation["eot20_level_m"] = predict_eot20(
    MOMBASA_LON,
    MOMBASA_LAT,
    validation["datetime"]
)

validation = validation.dropna(
    subset=["observed_level", "eot20_level_m"]
).copy()

validation["observed_centred_m"] = (
    validation["observed_level"]
    - validation["observed_level"].mean()
)

validation["eot20_centred_m"] = (
    validation["eot20_level_m"]
    - validation["eot20_level_m"].mean()
)

validation["residual_m"] = (
    validation["observed_centred_m"]
    - validation["eot20_centred_m"]
)

obs = validation["observed_centred_m"].to_numpy()
pred = validation["eot20_centred_m"].to_numpy()
residual = obs - pred

r, p = stats.pearsonr(obs, pred)
reg = stats.linregress(pred, obs)

metrics = pd.Series({
    "n": len(validation),
    "RMSE_m": np.sqrt(np.mean(residual**2)),
    "MAE_m": np.mean(np.abs(residual)),
    "mean_error_m": np.mean(residual),
    "Pearson_r": r,
    "Pearson_p": p,
    "R_squared": r**2,
    "regression_slope": reg.slope,
    "regression_intercept_m": reg.intercept,
})

display(metrics.to_frame("estimate"))

,estimate
n,8.403400e+04
RMSE_m,8.013765e-02
MAE_m,6.349708e-02
mean_error_m,2.563683e-16
Pearson_r,9.956336e-01
Pearson_p,0.000000e+00
R_squared,9.912863e-01
regression_slope,1.004041e+00
regression_intercept_m,2.563341e-16


## 6. Moving-block bootstrap confidence intervals

The bootstrap accounts for serial dependence by resampling contiguous 48-hour blocks. Candidate blocks are pooled across continuous gauge segments and sampled from the available block starts. This avoids giving short continuous segments the same selection probability as much longer segments.

In [6]:
def moving_block_bootstrap(
    df,
    block_length=48,
    n_boot=2000,
    confidence=0.95,
    gap_threshold_hours=1.5,
    random_seed=42,
):
    data = (
        df[
            [
                "datetime",
                "observed_centred_m",
                "eot20_centred_m",
            ]
        ]
        .dropna()
        .sort_values("datetime")
        .reset_index(drop=True)
    )

    gap_h = (
        data["datetime"]
        .diff()
        .dt.total_seconds()
        / 3600
    )

    data["segment_id"] = (
        gap_h.isna()
        | (gap_h > gap_threshold_hours)
    ).cumsum()

    blocks = []

    for _, seg in data.groupby("segment_id"):
        seg = seg.reset_index(drop=True)
        n_starts = len(seg) - block_length + 1

        if n_starts <= 0:
            continue

        for start in range(n_starts):
            blocks.append(
                (
                    seg["observed_centred_m"]
                    .iloc[start:start + block_length]
                    .to_numpy(),
                    seg["eot20_centred_m"]
                    .iloc[start:start + block_length]
                    .to_numpy(),
                )
            )

    if not blocks:
        raise ValueError(
            "No continuous segment is long enough "
            "for the selected block length."
        )

    rng = np.random.default_rng(random_seed)
    target_n = len(data)

    rmse_boot = np.empty(n_boot)
    mae_boot = np.empty(n_boot)
    r_boot = np.empty(n_boot)

    for b in range(n_boot):
        obs_parts = []
        pred_parts = []
        n = 0

        while n < target_n:
            obs_block, pred_block = blocks[
                rng.integers(0, len(blocks))
            ]
            obs_parts.append(obs_block)
            pred_parts.append(pred_block)
            n += len(obs_block)

        obs_b = np.concatenate(obs_parts)[:target_n]
        pred_b = np.concatenate(pred_parts)[:target_n]
        res_b = obs_b - pred_b

        rmse_boot[b] = np.sqrt(np.mean(res_b**2))
        mae_boot[b] = np.mean(np.abs(res_b))
        r_boot[b] = stats.pearsonr(obs_b, pred_b)[0]

    alpha = 1 - confidence
    lo = 100 * alpha / 2
    hi = 100 * (1 - alpha / 2)

    return pd.DataFrame({
        "metric": ["RMSE_m", "MAE_m", "Pearson_r"],
        "CI_lower": [
            np.percentile(rmse_boot, lo),
            np.percentile(mae_boot, lo),
            np.percentile(r_boot, lo),
        ],
        "CI_upper": [
            np.percentile(rmse_boot, hi),
            np.percentile(mae_boot, hi),
            np.percentile(r_boot, hi),
        ],
    })

bootstrap_ci = moving_block_bootstrap(validation)

validation_summary = (
    metrics.rename("estimate")
    .rename_axis("metric")
    .reset_index()
    .merge(
        bootstrap_ci,
        on="metric",
        how="left"
    )
)

display(validation_summary)

,metric,estimate,CI_lower,CI_upper
0,n,8.403400e+04,NaN,NaN
1,RMSE_m,8.013765e-02,0.078211,0.081659
2,MAE_m,6.349708e-02,0.062065,0.064671
3,mean_error_m,2.563683e-16,NaN,NaN
4,Pearson_r,9.956336e-01,0.995446,0.995852
5,Pearson_p,0.000000e+00,NaN,NaN
6,R_squared,9.912863e-01,NaN,NaN
7,regression_slope,1.004041e+00,NaN,NaN
8,regression_intercept_m,2.563341e-16,NaN,NaN


## 7. Predict EOT20 at Landsat scene acquisition times

These predictions are made at the Mombasa location. They characterize the tidal levels represented by the Landsat source-scene acquisition times. They should not be interpreted as 30 m tidal predictions or as a coast-wide tidal uncertainty field.

In [7]:
scene_tides = scenes.copy()

scene_tides["eot20_level_m"] = predict_eot20(
    MOMBASA_LON,
    MOMBASA_LAT,
    scene_tides["acquisition_datetime"]
)

scene_tides = scene_tides.dropna(
    subset=["eot20_level_m"]
).copy()

print("Scenes with valid tide predictions:",
      f"{len(scene_tides):,}")

display(
    scene_tides[
        [
            "year",
            "sensor",
            "scene_id",
            "acquisition_datetime",
            "eot20_level_m",
        ]
    ].head(10)
)

Scenes with valid tide predictions: 5,332


,year,sensor,scene_id,acquisition_datetime,eot20_level_m
0,2000,L7,LE07_L2SP_165061_20000115_20200918_02_T1,2000-01-15 07:17:27+00:00,0.557193
1,2000,L7,LE07_L2SP_165062_20000115_20200918_02_T1,2000-01-15 07:17:51+00:00,0.556488
2,2000,L7,LE07_L2SP_166061_20000122_20200918_02_T1,2000-01-22 07:23:37+00:00,-1.718353
3,2000,L7,LE07_L2SP_166062_20000122_20200918_02_T1,2000-01-22 07:24:01+00:00,-1.720563
4,2000,L7,LE07_L2SP_166063_20000122_20200918_02_T1,2000-01-22 07:24:25+00:00,-1.722754
5,2000,L7,LE07_L2SP_167061_20000129_20200918_02_T1,2000-01-29 07:29:48+00:00,0.335151
6,2000,L7,LE07_L2SP_167062_20000129_20200918_02_T1,2000-01-29 07:30:12+00:00,0.334674
7,2000,L7,LE07_L2SP_165062_20000131_20200918_02_T1,2000-01-31 07:17:50+00:00,-0.184332
8,2000,L7,LE07_L2SP_166061_20000223_20200918_02_T1,2000-02-23 07:23:25+00:00,-0.745957
9,2000,L7,LE07_L2SP_167061_20000301_20200918_02_T1,2000-03-01 07:29:38+00:00,-0.281892


## 8. Prepare the annual vertical tidal input

Because an annual median spectral composite has no single acquisition time, the tidal contribution is characterized from the distribution of EOT20 levels at all contributing Landsat scene times.

The annual vertical tidal uncertainty term is defined as half the central 95% range:

\[
U_{\eta,y}
=
\frac{
Q_{97.5}(\eta_y)-Q_{2.5}(\eta_y)
}{2}
\]

The resulting `U_eta_m` is the quantity carried forward to the horizontal tidal uncertainty calculation.

**Important:** this notebook currently predicts EOT20 only at Mombasa. Therefore the output below is the Mombasa vertical tidal input, not yet the final coast-wide \(U_{tide}\) input.

In [8]:
annual_tide_input = (
    scene_tides
    .groupby("year")["eot20_level_m"]
    .agg(
        scene_count="count",
        q025_tide_m=lambda x: x.quantile(0.025),
        median_tide_m="median",
        q975_tide_m=lambda x: x.quantile(0.975),
    )
    .reset_index()
)

annual_tide_input["central_95_range_m"] = (
    annual_tide_input["q975_tide_m"]
    - annual_tide_input["q025_tide_m"]
)

annual_tide_input["U_eta_m"] = (
    annual_tide_input["central_95_range_m"] / 2
)

annual_tide_input["location"] = "Mombasa"
annual_tide_input["longitude"] = MOMBASA_LON
annual_tide_input["latitude"] = MOMBASA_LAT
annual_tide_input["status"] = (
    "FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA"
)

annual_tide_input = annual_tide_input[
    [
        "year",
        "scene_count",
        "q025_tide_m",
        "median_tide_m",
        "q975_tide_m",
        "central_95_range_m",
        "U_eta_m",
        "location",
        "longitude",
        "latitude",
        "status",
    ]
]

display(annual_tide_input)

,year,scene_count,q025_tide_m,median_tide_m,q975_tide_m,central_95_range_m,U_eta_m,location,longitude,latitude,status
0,2000,81,-1.722754,-0.431250,0.605869,2.328623,1.164311,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
1,2001,88,-1.729640,-0.609864,0.772159,2.501799,1.250900,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
2,2002,79,-1.796021,-0.549121,0.540050,2.336071,1.168035,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
3,2003,77,-1.827919,-0.931895,0.548838,2.376757,1.188379,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
4,2004,84,-1.736606,-0.673141,0.695966,2.432571,1.216286,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
5,2005,91,-1.817701,-0.348444,0.664540,2.482241,1.241120,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
6,2006,69,-1.693794,-0.892707,0.480802,2.174596,1.087298,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
7,2007,98,-1.756657,-0.744188,0.609178,2.365835,1.182917,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
8,2008,95,-1.653056,-0.738700,0.692868,2.345924,1.172962,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA
9,2009,150,-1.640359,-0.653266,0.839999,2.480358,1.240179,Mombasa,39.6567,-4.07,FINAL_VERTICAL_INPUT_FOR_UTIDE_AT_MOMBASA


## 9. Export only the two tables to retain

The notebook exports only:

`Mombasa_EOT20_validation_summary.csv`  
A compact validation table for reporting and methodological documentation.

`Utide_Vertical_Tide_Input_Mombasa.csv`  
The clearly marked annual vertical tidal input. The `U_eta_m` field is the value carried forward to the later horizontal \(U_{tide}\) calculation at Mombasa.

The large paired validation table and the scene-by-scene EOT20 table are not exported because they can be reproduced from the preserved PSMSL source record, Landsat scene metadata and EOT20 model files.

In [9]:
validation_summary.to_csv(
    VALIDATION_SUMMARY_OUT,
    index=False
)

annual_tide_input.to_csv(
    UTIDE_VERTICAL_INPUT_OUT,
    index=False
)

print("KEEP THESE TWO DERIVED TABLES")
print("-----------------------------")
print("1. Validation/reporting:")
print("   ", VALIDATION_SUMMARY_OUT)
print()
print("2. FINAL MOMBASA VERTICAL INPUT FOR U_tide:")
print("   ", UTIDE_VERTICAL_INPUT_OUT)
print()
print(
    "Carry-forward field for U_tide calculation: U_eta_m"
)

KEEP THESE TWO DERIVED TABLES
-----------------------------
1. Validation/reporting:
    D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\EOT20_PMSL_tide\Mombasa_EOT20_validation_summary.csv

2. FINAL MOMBASA VERTICAL INPUT FOR U_tide:
    D:\1_GIt_Repositories\kenya-shoreline-change-2000-2024\Outputs\EOT20_PMSL_tide\Utide_Vertical_Tide_Input_Mombasa.csv

Carry-forward field for U_tide calculation: U_eta_m


## 10. Carry-forward to horizontal tidal uncertainty

The later slope notebook provides the local shoreline-intersection slope \(\beta_{i,y}\).

The Mombasa prototype combines the annual vertical tidal term with the local slope as:

\[
U_{tide,i,y}
=
\frac{U_{\eta,y}}
{\tan(\beta_{i,y})}
\]

For the full Kenyan coast, the next methodological step is to generate spatially appropriate EOT20 scene-time tidal summaries along the coast rather than applying the Mombasa `U_eta_m` values to all shoreline intersections.

## Table A1 (Supplementary Method S1)

In [10]:
table_a1 = validation_summary.copy()
for c in ["estimate", "CI_lower", "CI_upper"]:
    table_a1[c] = pd.to_numeric(table_a1[c], errors="coerce").round(4)
display(table_a1)
save_table(table_a1, "Table_A1_EOT20_validation_Mombasa")

,metric,estimate,CI_lower,CI_upper
0,n,84034.0000,NaN,NaN
1,RMSE_m,0.0801,0.0782,0.0817
2,MAE_m,0.0635,0.0621,0.0647
3,mean_error_m,0.0000,NaN,NaN
4,Pearson_r,0.9956,0.9954,0.9959
5,Pearson_p,0.0000,NaN,NaN
6,R_squared,0.9913,NaN,NaN
7,regression_slope,1.0040,NaN,NaN
8,regression_intercept_m,0.0000,NaN,NaN


Saved: Outputs\Tables\Table_A1_EOT20_validation_Mombasa.csv
